In [14]:
# test massimi di colonna nei dati storici
import sys
import os
import numpy as np
from scipy import stats

# Navigate from test/ to src/ using relative path
notebook_dir = os.path.dirname(os.path.abspath('.'))
src_path = os.path.join(notebook_dir,  'src')
sys.path.insert(0, src_path)

from common_lib import CommonLib
from stats_lib import DelaysStats

######################
def count_values_with_coverage(history_data, lower_bounds, upper_bounds):
    """
    Calculate coverage ratio within value ranges for each column.
    
    Uses vectorized NumPy operations for performance.
    
    Args:
        delays_summary: 2D array with 6 columns
        lower_bounds: Array of 6 lower bounds
        upper_bounds: Array of 6 upper bounds
    
    Returns:
        Array of 6 coverage ratios (0.0 to 1.0)
    """
    dim = history_data.shape[1]
    result = np.zeros(dim, dtype=np.float64)
    
    try:
        bounds_summary = np.asarray(history_data, dtype=np.int64)
        lower_bounds = np.asarray(lower_bounds, dtype=np.float64)
        upper_bounds = np.asarray(upper_bounds, dtype=np.float64)
        
        for col_idx in range(dim):
            column_data = bounds_summary[:, col_idx]
            
            unique, counts = np.unique(column_data, return_counts=True)
            
            max_value = int(unique.max()) if unique.size > 0 else 0
            count_array = np.zeros(max_value + 1, dtype=np.int64)
            
            count_array[unique] = counts
            
            min_idx = np.clip(np.round(lower_bounds[col_idx]).astype(np.int64), 0, len(count_array) - 1)
            max_idx = np.clip(np.round(upper_bounds[col_idx]).astype(np.int64), 0, len(count_array) - 1)
            
            if min_idx > max_idx:
                result[col_idx] = 0.0
                continue
            
            # Calculate the sum of counts within the specified range
            sum_range = np.sum(count_array[min_idx:max_idx + 1])
            # Calculate the total sum of counts for normalization
            total_sum = np.sum(count_array)
            
            # Calculate coverage ratio, handling division by zero
            result[col_idx] = sum_range / total_sum if total_sum > 0 else 0.0
        
        return result
        
    except (TypeError, ValueError, IndexError, OverflowError):
        return result

######################

data_file_name = "../data/estrazioni_2026_2006_copy.csv"
clib = CommonLib(data_file_name)
dlib = DelaysStats()

# Popolazione
history_data = clib.load_data() # Carica le estrazioni da un file CSV

minb = np.round(np.min(history_data, axis=0), 0)
maxb = np.round(np.max(history_data, axis=0), 0)
print(f"Bounds \n- minimo: {minb} \n- massimo: {maxb}")

u = np.round(np.mean(history_data, axis=0), 0)
x = np.round(np.std(history_data, axis=0), 0)
sk = np.round(stats.skew(history_data, axis=0), 2)

q1 = np.round(np.quantile(history_data, 0.25, axis=0), 0)
q3 = np.round(np.quantile(history_data, 0.75, axis=0), 0)

print(f"Bounds \n- media: {u} \n- deviazione standard: {x} \n- skewness: {sk} \n- q1: {q1} \n- q3: {q3}")

min_bounds_simm = np.round(u - x, 0)
max_bounds_simm = np.round(u + x, 0)
coverages_simm = np.round((count_values_with_coverage(history_data, min_bounds_simm, max_bounds_simm)) * 100, 2)
print(f"Bounds - copertura intervallo media ± deviazione standard: {coverages_simm}%")

min_bounds_asimm = q1
max_bounds_asimm = q3
coverages_asimm = np.round((count_values_with_coverage(history_data, min_bounds_asimm, max_bounds_asimm)) * 100, 2)
print(f"Bounds - copertura intervallo IQR: {coverages_asimm}%")

results = {}
results['description'] = "Bounds"
measures = {}
measures['values'] = [{}] * 4
for j in range(0, 4):
    type = ""
    match j:
        case 0: type = "Bounds primo estratto"
        case 1: type = "Bounds secondo estratto"
        case 2: type = "Bounds terzo estratto"
        case 3: type = "Bounds quarto estratto"
        case 4: type = "Bounds quinto estratto"
        case 5: type = "Bounds sesto estratto"
    measures['values'][j] = {
        'id': j,
        'type': type,
        'min': minb[j],
        'max': maxb[j],
        'mean': u[j],
        'std_dev': x[j],
        'skewness': sk[j],
        'q1': q1[j],
        'q3': q3[j],
        'simmetrica': {
            'min': min_bounds_simm[j],
            'max': max_bounds_simm[j],
            'coverage': coverages_simm[j]
        },
        'asimmetrica': {
            'min': min_bounds_asimm[j],
            'max': max_bounds_asimm[j],
            'coverage': coverages_asimm[j]
        }
    }
results['measures'] = measures
#results = dlib.get_delays_dict(history_data)
print(results)

Bounds 
- minimo: [ 1  2  4  9 14 26] 
- massimo: [65 82 85 87 89 90]
Bounds 
- media: [13. 26. 40. 53. 66. 78.] 
- deviazione standard: [11. 14. 15. 15. 14. 11.] 
- skewness: [ 1.24  0.58  0.14 -0.24 -0.65 -1.35] 
- q1: [ 5. 16. 28. 42. 57. 73.] 
- q3: [19. 35. 50. 64. 76. 86.]
Bounds - copertura intervallo media ± deviazione standard: [77.04 68.68 65.45 67.81 69.63 79.43]%
Bounds - copertura intervallo IQR: [52.08 50.78 51.4  52.54 50.38 53.27]%
{'description': 'Bounds', 'measures': {'values': [{'id': 0, 'type': 'Bounds primo estratto', 'min': np.int64(1), 'max': np.int64(65), 'mean': np.float64(13.0), 'std_dev': np.float64(11.0), 'skewness': np.float64(1.24), 'q1': np.float64(5.0), 'q3': np.float64(19.0), 'simmetrica': {'min': np.float64(2.0), 'max': np.float64(24.0), 'coverage': np.float64(77.04)}, 'asimmetrica': {'min': np.float64(5.0), 'max': np.float64(19.0), 'coverage': np.float64(52.08)}}, {'id': 1, 'type': 'Bounds secondo estratto', 'min': np.int64(2), 'max': np.int64(82), 'm